In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv("GROQ_API_KEY")

# load model 
from langchain_groq import ChatGroq
model = ChatGroq(model="llama-3.1-8b-instant", groq_api_key=groq_api_key)
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001712AB1AA20>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001712ABD7EF0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [8]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant. Answer all the questions to the best of your ability "),
        MessagesPlaceholder(variable_name="messages")
    ]

)

chain = prompt|model

In [10]:
# give human message
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi, My name is Pankaj and I am AI Engineer")])


AIMessage(content='Nice to meet you, Pankaj. As an AI Engineer, you must be working on exciting projects that involve the development and application of artificial intelligence and machine learning algorithms. What specific areas of AI are you interested in or currently working on? Is it computer vision, natural language processing, or perhaps robotics?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 48, 'total_tokens': 111, 'completion_time': 0.122003712, 'completion_tokens_details': None, 'prompt_time': 0.011833873, 'prompt_tokens_details': None, 'queue_time': 0.059270967, 'total_time': 0.133837585}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_ff2b098aaf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019bdf7c-c976-7ec2-b3b5-decad6d075c4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 48, 'output_tokens': 63, 'total_tokens

In [11]:
## message history

from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import  RunnableWithMessageHistory

store={}
def get_session_history(session_id:str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]


with_message_history = RunnableWithMessageHistory(chain,get_session_history)


In [12]:
config = {"configurable":{"session_id":"chat3"}}
response=with_message_history.invoke(
    [
         HumanMessage(content="Hi, My name is Pankaj and I am AI Engineer")
        
    ],
    config=config
)

response

AIMessage(content="Nice to meet you, Pankaj! As an AI Engineer, you work on designing, developing, and testing artificial intelligence and machine learning models. That's a fascinating field.\n\nWhat brings you here today? Do you have any specific questions or topics you'd like to discuss related to AI or engineering? I'm here to help.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 64, 'total_tokens': 132, 'completion_time': 0.162488198, 'completion_tokens_details': None, 'prompt_time': 0.004124768, 'prompt_tokens_details': None, 'queue_time': 0.050427852, 'total_time': 0.166612966}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019bdf80-4df1-78b2-a964-8d21c6abdbf8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 64, 'output_tokens': 68, 'total_tokens': 132})

## adding more complexity

In [13]:


prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant. Answer all the questions to the best of your ability in {langauge} "),
        MessagesPlaceholder(variable_name="messages")
    ]

)
chain = prompt|model

In [15]:
response_1= chain.invoke({"messages":[HumanMessage(content="Hi, My name is Pankaj and I am AI Engineer")],"langauge":"Hindi"})
response_1.content

'नमस्ते पंकज जी, मैं आपकी सहायता करने के लिए यहाँ हूँ। आपके पास AI Engineer के रूप में बहुत सारे अनुभव होंगे। क्या आपको कोई विशिष्ट समस्या या प्रश्न है जिसका समाधान मैं आपकी सहायता कर सकता हूँ?'

In [16]:
with_message_history=RunnableWithMessageHistory (
    chain,
    get_session_history,
    input_messages_key="messages"
)

In [18]:
config = {"configurable":{"session_id":"chat4"}}
response_2= chain.invoke(
                        {"messages":[HumanMessage(content="Hi, My name is Pankaj ")],"langauge":"Hindi"},
                        config=config
                        )
response_2.content

'नमस्ते पंकज जी, मैं आपकी सहायता करने के लिए यहाँ हूँ। आप क्या चाहते हैं?'